In [ ]:
import torch
import torch.nn as nn
import numpy as np    
import random
import ale_py
import gymnasium as gym
from collections import deque
import torch.optim as optim
from PIL import Image
import os
import cv2

from collections import deque
from IPython.display import Video

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
ENV_NAME = 'ALE/Enduro-v5'
GAMMA = 0.99
BATCH_SIZE = 32
BUFFER_SIZE = 50000
LR = 0.00025
EPSILON_START = 1.0
EPSILON_END = 0.1
EPSILON_DECAY = 250000
TARGET_UPDATE = 10000
MAX_EPISODES = 1000
WARMUP_STEPS = 50000
FRAME_SKIP = 4

In [ ]:
def preprocess_frame(frame):
    img = Image.fromarray(frame)
    img = img.convert('L')
    img = img.resize((84, 84), Image.Resampling.LANCZOS)
    return np.array(img, dtype=np.uint8)

In [ ]:
class ReplayBuffer:
    def __init__(self):
        self.memory = deque(maxlen=BUFFER_SIZE)

    def push(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def sample(self):
        batch = random.sample(self.memory, BATCH_SIZE)
        states, actions, rewards, next_states, dones = zip(*batch)

        s = torch.tensor(np.array(states), dtype=torch.float32).to(device)
        a = torch.tensor(actions, dtype=torch.long).to(device)
        r = torch.tensor(rewards, dtype=torch.float32).to(device)
        s_next = torch.tensor(np.array(next_states), dtype=torch.float32).to(device)
        d = torch.tensor(dones, dtype=torch.float32).to(device)

        return s, a, r, s_next, d

In [ ]:
class DQN(nn.Module):
    def __init__(self, num_actions):
        super(DQN, self).__init__()
        self.conv1 = nn.Conv2d(4, 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)
        
        self.fc1 = nn.Linear(3136, 512)
        self.fc2 = nn.Linear(512, num_actions)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x / 255.0
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

In [ ]:
class Agent:
    def __init__(self, num_actions):
        self.num_actions = num_actions
        self.steps_done = 0

        self.policy_net = DQN(num_actions).to(device)
        self.target_net = DQN(num_actions).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())

        self.optimizer = optim.RMSprop(self.policy_net.parameters(), lr=LR)
        self.loss_fn = nn.SmoothL1Loss()

    def select_action(self, state):
        epsilon = max(EPSILON_END, EPSILON_START - (self.steps_done / EPSILON_DECAY))

        if random.random() < epsilon:
            return random.randrange(self.num_actions), epsilon
        else:
            with torch.no_grad():
                state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
                q_values = self.policy_net(state_tensor)
                return q_values.argmax(1).item(), epsilon

    def optimize_model(self, buffer):
        if len(buffer.memory) < BATCH_SIZE:
            return

        s, a, r, s_next, d = buffer.sample()

        current_q = self.policy_net(s).gather(1, a.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            next_q = self.target_net(s_next).max(1)[0]
            target_q = r + (GAMMA * next_q * (1 - d))

        loss = self.loss_fn(current_q, target_q)
        
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()

        self.steps_done += 1

        if self.steps_done % TARGET_UPDATE == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

In [ ]:
env = gym.make(ENV_NAME, frameskip=FRAME_SKIP)
agent = Agent(env.action_space.n)
buffer = ReplayBuffer()

print("--- WARMUP PHASE ---")
raw_frame, _ = env.reset()
frames = deque([preprocess_frame(raw_frame)] * 4, maxlen=4)
current_state = np.array(frames, dtype=np.uint8)

for i in range(WARMUP_STEPS):
    action = env.action_space.sample()
    next_frame, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

    frames.append(preprocess_frame(next_frame))
    next_state = np.array(frames, dtype=np.uint8)

    reward_clip = np.sign(reward)
    buffer.push(current_state, action, reward_clip, next_state, done)
    
    current_state = next_state

    if done:
        raw_frame, _ = env.reset()
        frames = deque([preprocess_frame(raw_frame)] * 4, maxlen=4)
        current_state = np.array(frames, dtype=np.uint8)

    if (i + 1) % 5000 == 0:
        print(f"> Loaded {i + 1}/{WARMUP_STEPS} frames.")

print("\n--- TRAINING PHASE ---")

for episode in range(1, MAX_EPISODES + 1):
    raw_frame, _ = env.reset()
    frames = deque([preprocess_frame(raw_frame)] * 4, maxlen=4)
    current_state = np.array(frames, dtype=np.uint8)
    
    total_reward = 0
    done = False
    current_epsilon = 1.0

    while not done:
        action, current_epsilon = agent.select_action(current_state)
        
        next_frame, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        frames.append(preprocess_frame(next_frame))
        next_state = np.array(frames, dtype=np.uint8)

        reward_clip = np.sign(reward)
        buffer.push(current_state, action, reward_clip, next_state, done)
        
        agent.optimize_model(buffer)

        current_state = next_state
        total_reward += reward

    print(f"Episode: {episode}/{MAX_EPISODES} | Total Reward: {total_reward:.1f} | Epsilon: {current_epsilon:.4f}")

    if episode % 50 == 0:
        torch.save(agent.policy_net.state_dict(), f"model_enduro_ep{episode}.pth")
        print(f"--> Saved file: model_enduro_ep{episode}.pth")

env.close()
print("--- TRAINING COMPLETE ---")


In [ ]:
gym.register_envs(ale_py)

# ============================================================
# CONFIG
# ============================================================

MODEL_PATH = "/kaggle/input/models/phmngcbch/1312/pytorch/default/1/dqn_enduro_final.pth"
VIDEO_PATH = "/kaggle/working/enduro_test.mp4"
FPS = 30

# ============================================================
# CREATE ENVIRONMENT
# ============================================================

env = gym.make(
    ENV_NAME,
    frameskip=FRAME_SKIP,
    render_mode="rgb_array"
)

# ============================================================
# LOAD MODEL
# ============================================================

agent = Agent(env.action_space.n)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

agent.policy_net.load_state_dict(checkpoint)
agent.policy_net.eval()

print("Model loaded!")

# ============================================================
# RESET
# ============================================================

raw_frame, _ = env.reset()

frames_stack = deque(
    [preprocess_frame(raw_frame)] * 4,
    maxlen=4
)

state = np.array(frames_stack, dtype=np.uint8)

done = False
total_reward = 0
step = 0

video_frames = []

print("Testing...")

# ============================================================
# PLAY GAME
# ============================================================

while not done:

    # lưu frame RGB
    frame = env.render()
    video_frames.append(frame)

    state_tensor = (
        torch.tensor(state, dtype=torch.float32)
        .unsqueeze(0)
        .to(device)
    )

    with torch.no_grad():
        action = agent.policy_net(state_tensor).argmax(1).item()

    next_frame, reward, terminated, truncated, _ = env.step(action)

    done = terminated or truncated

    frames_stack.append(preprocess_frame(next_frame))
    state = np.array(frames_stack, dtype=np.uint8)

    total_reward += reward
    step += 1

print("=" * 50)
print("Episode Finished")
print("Steps :", step)
print("Reward:", total_reward)
print("=" * 50)

env.close()

# ============================================================
# SAVE VIDEO
# ============================================================

print("Saving video...")

height, width, _ = video_frames[0].shape

writer = cv2.VideoWriter(
    VIDEO_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    FPS,
    (width, height)
)

for frame in video_frames:
    writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

writer.release()

print("Video saved:", VIDEO_PATH)

# ============================================================
# SHOW VIDEO
# ============================================================

Video(VIDEO_PATH, embed=True)

In [ ]:
from IPython.display import Video, display

display(Video("/kaggle/working/enduro_test.mp4", embed=True))